# Tunnelling magnetoresistance: the spinor tunnelling problem, solved exactly

**Professor Nano Labs · Spintronics**

**Try it live in your browser:** [Tunnelling magnetoresistance lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/tunneling-magnetoresistance/)
**Watch first:** [Theory of the tunneling magnetoresistance](https://www.youtube.com/watch?v=wEox88pAC7Q)
**Watch later:** [Spintronics memories devices, how are information stored?](https://www.youtube.com/watch?v=ugrsqPkQXNY) · [How electricity actually works at the nanoscale?](https://www.youtube.com/watch?v=z7dVABiYnTk) · [Magnetization switching through spin transfer torque](https://www.youtube.com/watch?v=R9z5s225i1U)

The video's toy model: two free-electron ferromagnets with exchange-split bands (majority band bottom at 0, minority at $\Delta$), separated by a barrier of height $U$ and thickness $d$. The fixed layer's spins point along $\mathbf m_1=\hat z$, the free layer's along $\mathbf m_2(\theta,\varphi)$, so in the free layer the spinors are
$$|{+}\rangle=\begin{pmatrix}\cos\frac\theta2\\ e^{i\varphi}\sin\frac\theta2\end{pmatrix},\qquad |{-}\rangle=\begin{pmatrix}-e^{-i\varphi}\sin\frac\theta2\\ \cos\frac\theta2\end{pmatrix}.$$
The video solves it for a thick barrier and electrons arriving head on. Here we solve it **exactly** for every normal energy $E_x$ and sum all transverse modes with the **Landauer formula**,
$$\frac{G}{A}=\frac{e^2}{h}\frac{m}{2\pi\hbar^2}\int T(E_x)\,dE_x,\qquad \mathrm{TMR}=\frac{G_P-G_{AP}}{G_{AP}}.$$
Energies in eV, lengths in Å. It needs NumPy, SciPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.constants import hbar, m_e, e, h

C = hbar**2 / (2 * m_e) / e * 1e20                 # hbar^2/2m in eV A^2 (3.81)
G0 = e**2 / h * m_e / (2 * np.pi * hbar**2) * e * 1e-12   # (e^2/h)(m/2 pi hbar^2): S per eV per um^2
print(f'hbar^2/2m = {C:.4f} eV A^2;  (e^2/h)(m/2 pi hbar^2) = {G0:.2f} S per eV per um^2')

def spinors(th, ph):
    c, s = np.cos(th / 2), np.sin(th / 2)
    return np.array([[c, -np.exp(-1j * ph) * s], [np.exp(1j * ph) * s, c]])    # columns |+>, |->

# the Fe free-electron fit used by Slonczewski, k_up = 1.09 and k_dn = 0.42 1/A, rounded as in the lab
EF, D = 4.53, 3.85
print(f'Fe fit: E_F = {EF} eV, Delta = {D} eV: k_up = {np.sqrt(EF / C):.4f}, k_dn = {np.sqrt((EF - D) / C):.4f} 1/A')

## 1 · One electron: the exact continuum solution

In the barrier the potential does not depend on spin, so each spin component is $a_j e^{-\kappa x}+b_j e^{\kappa(x-d)}$ (scaled so that nothing overflows). An electron of spin $s$ comes in from the fixed layer with amplitude $1/\sqrt{k_s}$ (unit flux), is reflected into both spins, and leaves into the free layer's $|{\pm}\rangle$. Matching $\Psi$ and $\Psi'$ at $x=0$ and $x=d$ gives 8 linear equations for $r_\uparrow, r_\downarrow, a_\uparrow, b_\uparrow, a_\downarrow, b_\downarrow, c_+, c_-$ (the video's slides 20–22). The transmission into channel $\pm$ is the outgoing flux $k_\pm|c_\pm|^2$.

In [ ]:
def solve(Ex, EF, D, U, d, th, ph=0.0):
    '''Zero bias. Returns T[out, in] and the spin torque data for each injected spin.'''
    kL = [np.sqrt(complex(Ex) / C), np.sqrt(complex(Ex - D) / C)]
    kR = kL
    kap = np.sqrt((U - Ex) / C); ee = np.exp(-kap * d)
    R = spinors(th, ph)
    T = np.zeros((2, 2)); sols = []
    for s in range(2):
        if Ex <= [0, D][s]: sols.append(None); continue
        M = np.zeros((8, 8), complex); rhs = np.zeros(8, complex); amp = 1 / np.sqrt(kL[s].real)
        for j in range(2):
            inc = amp if j == s else 0
            M[2*j, j] = 1;          M[2*j, 2+2*j] = -1;    M[2*j, 3+2*j] = -ee;       rhs[2*j] = -inc
            M[2*j+1, j] = -1j*kL[j]; M[2*j+1, 2+2*j] = kap; M[2*j+1, 3+2*j] = -kap*ee; rhs[2*j+1] = -1j*kL[j]*inc
            M[4+2*j, 2+2*j] = ee;     M[4+2*j, 3+2*j] = 1
            M[5+2*j, 2+2*j] = -kap*ee; M[5+2*j, 3+2*j] = kap
            for m in range(2):
                M[4+2*j, 6+m] = -R[j, m]; M[5+2*j, 6+m] = -R[j, m] * 1j * kR[m]
        x = np.linalg.solve(M, rhs)
        for m in range(2):
            if Ex > [0, D][m]: T[m, s] = kR[m].real * abs(x[6+m])**2
        sols.append(x)
    return T, sols, kap

def video_T(Ex, D, U, d, th):
    '''The video's thick-barrier coefficients C_up, C_dn (slide 22), summed over both incoming spins.'''
    k = [np.sqrt(Ex / C), np.sqrt(max(Ex - D, 0) / C)]; kap = np.sqrt((U - Ex) / C)
    A = [4 * kk * kap / (kap**2 + kk**2) for kk in k]
    return np.exp(-2 * kap * d) * (np.cos(th/2)**2 * (A[0]**2 + A[1]**2) + np.sin(th/2)**2 * 2 * A[0] * A[1])

U = EF + 3.0
for d in (3, 5, 8, 12):
    for th in (0, np.pi):
        Tex = solve(EF, EF, D, U, d, th)[0].sum()
        print(f'd = {d:2d} A, theta = {th:4.2f}: exact T = {Tex:.4e}, video thick-barrier = {video_T(EF, D, U, d, th):.4e}, ratio {video_T(EF, D, U, d, th) / Tex:.6f}')

The thick-barrier formula is excellent once the barrier is a few ångströms: multiple reflections inside the barrier, which it neglects, are suppressed by $e^{-2\kappa d}$. Now the wavefunction itself: incoming and reflected waves interfere in the fixed layer, the wave decays exponentially in the barrier, and comes out as the free layer's majority and minority waves.

In [ ]:
def psi(x, Ex, EF, D, U, d, th, s, sol):
    kL = [np.sqrt(complex(Ex) / C), np.sqrt(complex(Ex - D) / C)]; kap = np.sqrt((U - Ex) / C); R = spinors(th, 0)
    amp = 1 / np.sqrt(kL[s].real)
    out = np.zeros((len(x), 2), complex)
    for i, xx in enumerate(x):
        if xx < 0:
            v = np.array([sol[j] * np.exp(-1j * kL[j] * xx) for j in range(2)]); v[s] += amp * np.exp(1j * kL[s] * xx)
        elif xx <= d:
            v = np.array([sol[2+2*j] * np.exp(-kap * xx) + sol[3+2*j] * np.exp(kap * (xx - d)) for j in range(2)])
        else:
            v = np.array([sol[6+m] * np.exp(1j * kL[m] * (xx - d)) for m in range(2)])     # along m2
        out[i] = v
    return out

d, th = 8.0, np.pi / 3
T, sols, kap = solve(EF, EF, D, U, d, th)
x = np.linspace(-8, d + 8, 1200)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for s, name in enumerate(('majority', 'minority')):
    w = psi(x, EF, EF, D, U, d, th, s, sols[s]) * np.sqrt(np.sqrt(EF - [0, D][s]) / np.sqrt(C))   # amplitude-1 incoming wave
    ax[s].semilogy(x, abs(w[:, 0])**2, 'tab:red', label='majority part'); ax[s].semilogy(x, abs(w[:, 1])**2, 'tab:blue', label='minority part')
    ax[s].axvspan(0, d, color='0.85'); ax[s].set(title=f'{name} electron, theta = 60 deg', xlabel='x (A)', ylabel='|psi|^2', ylim=(1e-8, 10)); ax[s].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2 · Every electron: the Landauer sum

With the same mass everywhere, the transmission depends on $\mathbf k_\parallel$ only through $E_x=E_F-\hbar^2k_\parallel^2/2m$, so the sum over transverse modes is one integral over $E_x$. Most of it comes from a narrow cone near normal incidence: $\kappa^2=\kappa_0^2+k_\parallel^2$ grows with $k_\parallel$.

In [ ]:
def Ttot(Ex, EF, D, U, d, th):
    return solve(Ex, EF, D, U, d, th)[0].sum()

def conductance(EF, D, phiB, d, th):
    U = EF + phiB; kap = np.sqrt(phiB / C); de = C * kap / d
    pts = sorted({p for p in [0, D, EF] + [EF - de * 2**j for j in range(-1, 9)] if 0 <= p <= EF})
    return G0 * sum(quad(Ttot, a, b, args=(EF, D, U, d, th), epsrel=1e-9, limit=200)[0] for a, b in zip(pts[:-1], pts[1:]))

def tmr_video(EF, D, phiB):
    ku, kd = np.sqrt(EF / C), np.sqrt(max(EF - D, 0) / C); k2 = phiB / C
    PJ = (ku - kd) / (ku + kd); Ps = PJ * (k2 - ku * kd) / (k2 + ku * kd)
    return 2 * Ps**2 / (1 - Ps**2), 2 * PJ**2 / (1 - PJ**2), Ps, PJ

phiB, d = 3.0, 8.0
GP, GAP = conductance(EF, D, phiB, d, 0), conductance(EF, D, phiB, d, np.pi)
tv, tj, Ps, PJ = tmr_video(EF, D, phiB)
print(f'RA_P = {1/GP:.4g} Ohm um^2, RA_AP = {1/GAP:.4g} Ohm um^2')
print(f'TMR exact = {100*(GP/GAP-1):.2f}%,  video (k_par = 0, thick) = {100*tv:.2f}% (P = {Ps:.3f}),  Julliere = {100*tj:.2f}% (P = {PJ:.3f})')

kp = np.linspace(0, np.sqrt(EF / C) * 0.999, 200)
dG = [[Ttot(EF - C * k**2, EF, D, U, d, th) * k for k in kp] for th in (0, np.pi)]
plt.figure(figsize=(5.5, 3.4))
plt.plot(kp, dG[0] / max(dG[0]), 'tab:red', label='parallel'); plt.plot(kp, dG[1] / max(dG[0]), 'tab:blue', label='antiparallel')
plt.axvline(np.sqrt((EF - D) / C), ls='--', c='tab:blue', lw=1); plt.xlabel('k_par (1/A)'); plt.ylabel('dG/dk_par (scaled)'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
ds = np.arange(3, 26, 1.0)
tm = [100 * (conductance(EF, D, 3.0, dd, 0) / conductance(EF, D, 3.0, dd, np.pi) - 1) for dd in ds]
hs = np.linspace(0.3, 5, 24)
th_ = [100 * (conductance(EF, D, hh, 8.0, 0) / conductance(EF, D, hh, 8.0, np.pi) - 1) for hh in hs]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(ds, tm, label='exact'); ax[0].axhline(100 * tv, ls='--', c='0.5', label='video, k_par = 0'); ax[0].axhline(100 * tj, ls='--', c='tab:orange', label='Julliere')
ax[0].set(xlabel='d (A)', ylabel='TMR (%)', title='barrier 3 eV'); ax[0].legend()
ax[1].plot(hs, th_, label='exact'); ax[1].plot(hs, [100 * tmr_video(EF, D, hh)[0] for hh in hs], '--', c='0.5', label='video, k_par = 0'); ax[1].axhline(100 * tj, ls='--', c='tab:orange', label='Julliere')
ax[1].set(xlabel='U - E_F (eV)', ylabel='TMR (%)', title='d = 8 A'); ax[1].legend()
plt.tight_layout(); plt.show()

## 3 · Turning the free layer

The conductance against $\theta$, its four spin channels, and the spin torque. In the barrier, which has no exchange field, the spin current $\mathbf j_s=\frac{\hbar}{m}\mathrm{Im}[\Psi^\dagger\boldsymbol\sigma\Psi']$ is conserved. Divided by the particle current, its component across $\mathbf m_2$ in the plane of $\mathbf m_1,\mathbf m_2$ is the spin (in units of $\hbar/2$) each tunnelling electron hands to the free layer: the in-plane spin-transfer torque per electron.

In [ ]:
sig = [np.array([[0, 1], [1, 0]]), np.array([[0, -1j], [1j, 0]]), np.diag([1, -1])]
def spin_current(Ex, EF, D, U, d, th, ph=0.0):
    T, sols, kap = solve(Ex, EF, D, U, d, th, ph)
    x = d / 2; js = np.zeros(3); jp = 0
    for s, sol in enumerate(sols):
        if sol is None: continue
        v = np.array([sol[2+2*j] * np.exp(-kap * x) + sol[3+2*j] * np.exp(kap * (x - d)) for j in range(2)])
        dv = np.array([-kap * sol[2+2*j] * np.exp(-kap * x) + kap * sol[3+2*j] * np.exp(kap * (x - d)) for j in range(2)])
        js += [np.imag(v.conj() @ S @ dv) for S in sig]; jp += np.imag(v.conj() @ dv)
    return js, jp, T

def angle_data(EF, D, phiB, d, th, ph=0.0):
    U = EF + phiB; kap = np.sqrt(phiB / C); de = C * kap / d
    pts = sorted({p for p in [0, D, EF] + [EF - de * 2**j for j in range(-1, 9)] if 0 <= p <= EF})
    xg, wg = np.polynomial.legendre.leggauss(16)
    G = 0; ch = np.zeros((2, 2)); js = np.zeros(3); jp = 0
    for a, b in zip(pts[:-1], pts[1:]):
        for xx, ww in zip(xg, wg):
            E = a + (b - a) * (xx + 1) / 2; w = (b - a) / 2 * ww
            j, p, T = spin_current(E, EF, D, U, d, th, ph)
            ch += G0 * w * T; js += w * j; jp += w * p
    m2 = np.array([np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)]); m1 = np.array([0, 0, 1.0])
    e_in = np.cross(m2, np.cross(m1, m2)); e_in = e_in / np.linalg.norm(e_in) if np.linalg.norm(e_in) > 1e-12 else e_in
    return ch.sum(), ch, (js / jp) @ e_in, (js / jp) @ m2

ths = np.linspace(0, np.pi, 19)
res = [angle_data(EF, D, 3.0, 8.0, t) for t in ths]
Gt = np.array([r[0] for r in res])
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].plot(np.degrees(ths), Gt / Gt[0], 'o-', label='exact'); ax[0].plot(np.degrees(ths), np.cos(ths/2)**2 + np.sin(ths/2)**2 * Gt[-1] / Gt[0], 'k--', lw=1, label='G_P cos^2 + G_AP sin^2')
ax[0].set(xlabel='theta (deg)', ylabel='G/G_P'); ax[0].legend(fontsize=8)
for (o, s), lab, st in [((0, 0), 'maj->maj', 'r-'), ((1, 0), 'maj->min', 'r--'), ((1, 1), 'min->min', 'b-'), ((0, 1), 'min->maj', 'b--')]:
    ax[1].plot(np.degrees(ths), [r[1][o, s] / Gt[0] for r in res], st, label=lab)
ax[1].set(xlabel='theta (deg)', ylabel='channel / G_P'); ax[1].legend(fontsize=8)
ax[2].plot(np.degrees(ths), [r[2] for r in res]); ax[2].set(xlabel='theta (deg)', ylabel='in-plane torque per electron (hbar/2)')
plt.tight_layout(); plt.show()
print(f'cos-law check at 90 deg: G(90)/mean(G_P, G_AP) - 1 = {Gt[9] / ((Gt[0] + Gt[-1]) / 2) - 1:.2e}')

## 4 · Bias, with a second method: the lattice Green's function

Under a bias the barrier is a trapezoid. The lab handles any potential with Green's functions on a fine lattice (spacing $a$), with the leads as exact self-energies, as in the quantum transport labs. We check that method against the exact continuum solution at zero bias first, then use it for the current
$$J=\frac{e}{h}\frac{m}{2\pi\hbar^2}\int T(E_x;V)\,w(E_x)\,dE_x,\qquad w=\text{width of }[E_F-eV,E_F]\text{ above }E_x .$$

In [ ]:
def lead(E, eb, t0):
    x = (E - eb) / (4 * t0)
    if x >= 0: ka = 2 * np.arcsin(np.sqrt(x)); return -t0 * np.exp(1j * ka), 2 * t0 * np.sin(ka)
    return -t0 * np.exp(-2 * np.arcsinh(np.sqrt(-x))), 0.0

def lattice_T(Ex, EF, D, U, d, th, V=0.0, a=0.025):
    t0 = C / a**2; N = int(round(d / a)); n = np.arange(1, N + 1); h_ = U - V * (n - 0.5) / N + 2 * t0
    S1 = [lead(Ex, 0, t0), lead(Ex, D, t0)]; S2 = [lead(Ex, -V, t0), lead(Ex, D - V, t0)]
    g = np.zeros(2, complex); gN1 = np.zeros(2, complex)
    for s in range(2):
        gg = 1 / (Ex - h_[0] - S1[s][0]); pr = gg
        for k in range(1, N): gg = 1 / (Ex - h_[k] - t0**2 * gg); pr = gg * (-t0) * pr
        g[s], gN1[s] = gg, pr
    R = spinors(th, 0); Sig2 = R @ np.diag([S2[0][0], S2[1][0]]) @ R.conj().T
    GN1 = np.linalg.inv(np.diag(1 / g) - Sig2) @ np.diag(gN1 / g)
    A = R.conj().T @ GN1
    return sum(S2[o][1] * S1[s][1] * abs(A[o, s])**2 for o in range(2) for s in range(2))

for th in (0, np.pi / 2, np.pi):
    a_, b_ = Ttot(EF, EF, D, U, 8.0, th), lattice_T(EF, EF, D, U, 8.0, th)
    print(f'theta = {th:.2f}: continuum {a_:.6e}, lattice {b_:.6e}, difference {b_ / a_ - 1:+.1e}')

def current(EF, D, phiB, d, V, th):
    U = EF + phiB; kap = np.sqrt(phiB / C); de = C * kap / d
    pts = sorted({p for p in [0, D, D - V, EF - V, EF] + [EF - de * 2**j for j in range(-1, 9)] if 0 <= p <= EF})
    f = lambda E: lattice_T(E, EF, D, U, d, th, V) * max(0.0, EF - max(E, EF - V))
    return G0 * sum(quad(f, a, b, epsrel=1e-7, limit=100)[0] for a, b in zip(pts[:-1], pts[1:]))   # A per um^2

Vs = np.arange(0.1, 1.51, 0.1)
tmrV = [100 * (current(EF, D, 3.0, 8.0, v, 0) / current(EF, D, 3.0, 8.0, v, np.pi) - 1) for v in Vs]
plt.figure(figsize=(5.5, 3.4)); plt.plot(np.r_[0, Vs], np.r_[100 * (GP / GAP - 1), tmrV], 'o-')
plt.xlabel('bias V (volts)'); plt.ylabel('TMR (%)'); plt.tight_layout(); plt.show()

## Checks: the numbers quoted in the lab's answers

The lab works on the lattice (spacing 0.025 Å); this notebook mostly in the continuum. They agree to a few parts in a thousand, which is the tolerance used below.

In [ ]:
close = lambda a, b, tol=0.005: abs(a / b - 1) < tol
# Q1: head on, parallel, Fe fit, 3 eV, 8 A: majority 2.6e-6, minority 1.6e-6, ratio 1.59; kappa 0.89 1/A
T, _, kap = solve(EF, EF, D, EF + 3, 8.0, 0)
assert close(T[:, 0].sum(), 2.62e-6) and close(T[:, 1].sum(), 1.645e-6) and abs(T[:, 0].sum() / T[:, 1].sum() - 1.59) < 0.01 and abs(kap - 0.887) < 0.001
# Q2: 8 -> 10 A divides the head-on transmission by 34.8
r = Ttot(EF, EF, D, EF + 3, 8.0, 0) / Ttot(EF, EF, D, EF + 3, 10.0, 0)
assert abs(r - 34.8) < 0.1
# Q3: TMR exact 39%, video 2.7%, Julliere 48%; RA_P 7.8e3 Ohm um^2
assert abs(100 * (GP / GAP - 1) - 39.1) < 0.3 and abs(100 * tv - 2.71) < 0.01 and abs(100 * tj - 48.4) < 0.1 and close(1 / GP, 7830, 0.01)
# Q4: 9.6% at 20 A, 4.6% at 40 A
t20 = 100 * (conductance(EF, D, 3, 20, 0) / conductance(EF, D, 3, 20, np.pi) - 1)
t40 = 100 * (conductance(EF, D, 3, 40, 0) / conductance(EF, D, 3, 40, np.pi) - 1)
assert abs(t20 - 9.6) < 0.15 and abs(t40 - 4.6) < 0.15
# Q5: strongly polarized (E_F 3, Delta 2.8), 2 eV, 8 A: 217% exact, Julliere 107%, video 14.5%; cone 0.30 > k_dn 0.23
g1, g2 = conductance(3.0, 2.8, 2.0, 8.0, 0), conductance(3.0, 2.8, 2.0, 8.0, np.pi)
tv5, tj5, _, _ = tmr_video(3.0, 2.8, 2.0)
assert abs(100 * (g1 / g2 - 1) - 217) < 2 and abs(100 * tj5 - 106.6) < 0.2 and abs(100 * tv5 - 14.5) < 0.1
assert abs(np.sqrt(np.sqrt(2 / C) / 8) - 0.30) < 0.005 and abs(np.sqrt(0.2 / C) - 0.229) < 0.001
# Q6: 90 deg: G = 0.859 G_P = mean of G_P and G_AP (0.719 G_P), to better than 1e-6
G90, ch90, tq90, _ = angle_data(EF, D, 3.0, 8.0, np.pi / 2)
assert abs(G90 / GP - 0.859) < 0.002 and abs(GAP / GP - 0.719) < 0.002 and abs(G90 / ((GP + GAP) / 2) - 1) < 1e-5
assert abs(tq90 - 0.31) < 0.01
# Q7: half metal (E_F 3, Delta 3.5): G_AP = 0, G(90) = G_P/2, torque per electron 1.00
Gh0 = angle_data(3.0, 3.5, 3.0, 8.0, 0)[0]; Gh90, _, tqh, _ = angle_data(3.0, 3.5, 3.0, 8.0, np.pi / 2)
assert conductance(3.0, 3.5, 3.0, 8.0, np.pi) < 1e-30 * Gh0 + 1e-300 and abs(Gh90 / Gh0 - 0.5) < 1e-5 and abs(tqh - 1.0) < 0.005
# Q8: TMR 31.7% at 0.5 V, halving near 0.92 V
J = lambda v, th: current(EF, D, 3.0, 8.0, v, th)
t05 = 100 * (J(0.5, 0) / J(0.5, np.pi) - 1)
lo, hi = 0.8, 1.0
for _ in range(10):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if 100 * (J(mid, 0) / J(mid, np.pi) - 1) > 100 * (GP / GAP - 1) / 2 else (lo, mid)
assert abs(t05 - 31.7) < 0.3 and abs((lo + hi) / 2 - 0.92) < 0.01
print(f"Q8: TMR(0.5 V) = {t05:.1f}%, halves at {(lo + hi) / 2:.3f} V")
print("all the lab's answers check out")

## Things to try

1. **The barrier's effective mass.** In MgO the tunnelling electrons have $m^*\approx0.4\,m_e$; the boundary condition becomes continuity of $\psi'/m$. How does that change Slonczewski's $P$ and the exact TMR?
2. **Different electrodes.** Make the two ferromagnets different (two values of $\Delta$). Can the TMR become negative, as Slonczewski predicted when one tunnelling polarization changes sign?
3. **The field-like torque.** Compute the out-of-plane part of the spin current for electrons injected from both sides and show that it cancels at linear order in the bias for identical electrodes, but not at second order.
4. **Temperature.** Replace the window $w(E_x)$ by Fermi functions and see how much the zero-bias TMR changes at 300 K.